In [0]:
orders_df = spark.read.table(
    "db_project_m_catalog.db_olist_dataset.orders"
)

display(orders_df)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
""")

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

Bronze Layer


In [0]:
orders_df.write.mode("overwrite").saveAsTable("workspace.bronze.orders")

In [0]:
customer_df = spark.read.table(
    "db_project_m_catalog.db_olist_dataset.customers"
)

customer_df.write.mode("overwrite").saveAsTable("workspace.bronze.customers")


In [0]:
products_df = spark.read\
    .option("header", "true")\
    .table(
    "db_project_m_catalog.db_olist_dataset.products"
)

products_df.write.mode("overwrite").saveAsTable("workspace.bronze.products")

In [0]:
geolocation_df = spark .read.table("db_project_m_catalog.db_olist_dataset.geolocation")

geolocation_df.write.mode("overwrite").saveAsTable("workspace.bronze.geolocation")

In [0]:
order_items_df = spark.read.table("db_project_m_catalog.db_olist_dataset.order_items")

order_items_df.write.mode("overwrite").saveAsTable("workspace.bronze.order_items")


In [0]:
order_payments_df = spark.read.table("db_project_m_catalog.db_olist_dataset.order_payments")

order_payments_df.write.mode("overwrite").saveAsTable("workspace.bronze.order_payments")

In [0]:
order_reviews_df = spark.read.table("db_project_m_catalog.db_olist_dataset.order_reviews")

order_reviews_df.write.mode("overwrite").saveAsTable("workspace.bronze.order_reviews")


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.bronze.product_category_name_translation AS

SELECT
    string_field_0 AS product_category_name,
    string_field_1 AS product_category_name_english

FROM db_project_m_catalog.db_olist_dataset.p_category_name_translation

WHERE string_field_0 != 'product_category_name';

In [0]:
seller_df = spark.read.table("db_project_m_catalog.db_olist_dataset.sellers")

seller_df.write.mode("overwrite").saveAsTable("workspace.bronze.sellers")

Pre Silver Reviews


In [0]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM workspace.bronze.orders
""").show()

In [0]:
bronze_orders =  spark.read.table("workspace.bronze.orders")
bronze_ordersworkspace.bronze.order_reviews.show()

In [0]:
bronze_product = spark.read.table("workspace.bronze.products")
bronze_product.show()


In [0]:
bronze_geo = spark.read.table("workspace.bronze.geolocation")
bronze_geo.show()
bronze_seller = spark.read.table("workspace.bronze.sellers")
bronze_seller.show()

In [0]:
p_category_df.show()

In [0]:
order_reviews_df.show()

In [0]:
order_payments_df.show()

In [0]:
order_items_df.show()

In [0]:
customer_df= spark.read.table("workspace.bronze.customers")
customer_df.show()


In [0]:
from pyspark.sql import functions as F
customer_df.select(F.col("customer_state")).distinct().show()

In [0]:
customer_df.printSchema()

In [0]:
customer_df.groupBy("customer_id")\
    .count()\
    .withColumnRenamed("count","id_count")\
    .filter(F.col("id_count")>1)\
    .show()

In [0]:
customer_df.filter(F.col("customer_zip_code_prefix").isNull()).show()

In [0]:
customer_df = customer_df \
    .withColumn("customer_state", F.initcap(F.col("customer_state"))) \
    .withColumn("customer_city", F.initcap(F.col("customer_city")))

customer_df.show()

In [0]:
silver_customer_df = customer_df.dropDuplicates(["customer_id"])
silver_customer_df.show()
silver_customer_df.count()
silver_customer_df.printSchema()
silver_customer_df.write.mode("overwrite").saveAsTable("workspace.silver.silver_customer")


In [0]:
geolocation_df = spark.read.table("workspace.bronze.geolocation")
geolocation_df.show()


In [0]:
geolocation_df.select("geolocation_city").distinct().show()


In [0]:
geo_temp_df = geolocation_df.withColumn("geolocation_city", F.lower(F.col("geolocation_city")))
geo_temp_df.select("geolocation_city").distinct().show()

In [0]:
geo_temp_df.groupBy("geolocation_city")\
    .count()\
    .orderBy(F.desc("count"))\
    .display()

In [0]:
city_dict = {
    # São Paulo
    "sao paulo": "Sao Paulo",
    "são paulo": "Sao Paulo",
    "saopaulo": "Sao Paulo",
    "são paulo ": "Sao Paulo",
    "sao  paulo": "Sao Paulo",

    # Rio de Janeiro
    "rio de janeiro": "Rio de Janeiro",
    "riodejaneiro": "Rio de Janeiro",
    "rio-de-janeiro": "Rio de Janeiro",
    "rio  de janeiro": "Rio de Janeiro",

    # Belo Horizonte
    "belo horizonte": "Belo Horizonte",
    "belohorizonte": "Belo Horizonte",
    "belo-horizonte": "Belo Horizonte",
    "belo  horizonte": "Belo Horizonte",

    # Curitiba
    "curitiba": "Curitiba",
    "curitba": "Curitiba",
    "curitibia": "Curitiba",
    "curitiba ": "Curitiba",

    # Porto Alegre
    "porto alegre": "Porto Alegre",
    "portoalegre": "Porto Alegre",
    "porto-alegre": "Porto Alegre",
    "porto  alegre": "Porto Alegre",

    # Salvador
    "salvador": "Salvador",
    "salvado": "Salvador",
    "salvador da bahia": "Salvador",
    "salvador-da-bahia": "Salvador",

    # Guarulhos
    "guarulhos": "Guarulhos",
    "guarulhos sp": "Guarulhos",
    "guarulhos-sp": "Guarulhos",
    "guarulhos ": "Guarulhos",

    # Brasília
    "brasilia": "Brasilia",
    "brasília": "Brasilia",
    "brasilia df": "Brasilia",
    "brasilia-df": "Brasilia",

    # São Bernardo do Campo
    "sao bernardo do campo": "Sao Bernardo do Campo",
    "são bernardo do campo": "Sao Bernardo do Campo",
    "saobernardo do campo": "Sao Bernardo do Campo",
    "sao bernardo docampo": "Sao Bernardo do Campo",
    "sao bernardo do  campo": "Sao Bernardo do Campo"
}

geo_temp_df = geo_temp_df.replace(
    city_dict,
    subset=["geolocation_city"]
)

In [0]:
geo_temp_df.groupBy("geolocation_city")\
    .count()\
    .orderBy(F.desc("count"))\
    .display()

In [0]:
geo_temp_df.display()

In [0]:
geo_temp_df.groupBy("geolocation_zip_code_prefix")\
    .count()\
    .display()

In [0]:
geo_temp_df.write.mode("overwrite").saveAsTable("workspace.silver.silver_geo")


In [0]:
o_item_temp = spark.read.table("workspace.bronze.order_items")
o_item_temp.display()

In [0]:
o_item_temp.printSchema()

In [0]:
oi1_df = o_item_temp.filter(F.col("price") < 0)
oi1_df.display()

In [0]:
o_item_temp.write.mode("overwrite").saveAsTable("workspace.silver.silver_order_items")

In [0]:
o_pay_temp = spark.read.table("workspace.bronze.order_payments")
o_pay_temp.display()

In [0]:
o_pay_temp.select("payment_type").distinct().display()

In [0]:
payment_dict = {
    "credit_card": "Credit Card",
    "boleto": "Bank Payment",
    "voucher": "Voucher",
    "debit_card": "Debit Card",
    "not_defined": "Unknown"
}
o_pay_temp = o_pay_temp.replace(payment_dict, subset=["payment_type"])
o_pay_temp.display()


In [0]:
o_pay_temp.write.mode("overwrite").saveAsTable("workspace.silver.silver_order_payments")


In [0]:
o_reviews_temp = spark.read.table("workspace.bronze.order_reviews")
o_reviews_temp.display()

In [0]:
from pyspark.sql import functions as F
o_reviews_temp.filter(F.col("order_id").isNull()).select("order_id").display()

In [0]:
o_reviews_temp.write.mode("overwrite").saveAsTable("workspace.silver.silver_order_reviews")


In [0]:
orders_temp = spark.read.table("workspace.bronze.orders")
orders_temp.display()


In [0]:
orders_temp.select("order_status").distinct().display()

In [0]:
from pyspark.sql import functions as F
orders_temp = orders_temp.withColumn("order_status",F.initcap(F.col("order_status")))
orders_temp.display()


In [0]:
o_3 = orders_temp.groupBy("order_id")\
    .count()\
    .filter("count > 1")\
    .display()

In [0]:
from pyspark.sql import functions as F

orders_temp = orders_temp.withColumn(
    "order_cancellation_stage",
    F.when(
        F.col("order_status") != "Canceled",
        "Not Canceled"
    )
    .when(
        F.col("order_approved_at").isNull(),
        "Before Approval"
    )
    .when(
        F.col("order_delivered_carrier_date").isNull(),
        "After Approval"
    )
    .when(
        F.col("order_delivered_customer_date").isNull(),
        "During Delivery"
    )
    .otherwise("Delivered")
)

In [0]:
orders_temp.display()

In [0]:
orders_temp.write.mode("overwrite").saveAsTable("workspace.silver.silver_orders")

In [0]:
p_c_name_temp = spark.read.table(
    "workspace.bronze.product_category_name_translation"
)

p_c_name_temp.show()


In [0]:
products_temp = spark.read.table("workspace.bronze.products")
products_temp.show()

In [0]:
product_f1 = products_temp.join(
    p_c_name_temp,
    on = "product_category_name",
    how = "left"
)
product_f1.show()
product_f1 = product_f1.drop("product_category_name")
product_f1 = product_f1.withColumnRenamed("product_category_name_english", "product_category_name")


In [0]:
product_f1.display()

In [0]:
product_f1.printSchema()

In [0]:
product_f1.display()

In [0]:
from pyspark.sql import functions as F
product_f1 = product_f1\
    .withColumn("product_name_lenght", F.col("product_name_lenght").cast("double"))\
    .withColumn("product_description_lenght", F.col("product_description_lenght").cast("double"))\
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast("double"))\
    .withColumn("product_weight_g", F.col("product_weight_g").cast("double"))\
    .withColumn("product_length_cm", F.col("product_length_cm").cast("double"))\
    .withColumn("product_height_cm", F.col("product_height_cm").cast("double"))\
    .withColumn("product_width_cm", F.col("product_width_cm").cast("double"))
product_f1.printSchema()


In [0]:
pfinal_test1 = product_f1\
    .fillna(0, subset=["product_description_lenght"])\
    .fillna(0, subset=["product_name_lenght"])\
    .fillna(0, subset=["product_photos_qty"])\
    .fillna("No Category", subset=["product_category_name"])
pfinal_test1.printSchema()

pfinal_test1.display()

                            

In [0]:
pfinal_test1 = pfinal_test1\
    .replace({"Baby": "Baby Care"}, subset=["product_category_name"])\
    .withColumn("product_category_name", F.regexp_replace("product_category_name", "_", " "))\
    .withColumn("product_category_name", F.initcap(F.col("product_category_name")))
pfinal_test1.display()


In [0]:
pfinal_test1.write.mode("overwrite").saveAsTable("workspace.silver.silver_product")

In [0]:
seller_temp = spark.read.table("workspace.bronze.sellers")
seller_temp.show()
seller_temp.printSchema()
seller_temp.display()

In [0]:
seller_1 = seller_temp.groupBy("seller_id").count().orderBy(F.desc("count"))

seller_1.show()

In [0]:
seller2 = seller_temp.select(F.col("seller_city")).distinct()
seller2.display()

In [0]:
city_dict_2 = {
    "sao paulo sp": "Sao Paulo",
    "sp / sp": "Sao Paulo",
    "sao pauo": "Sao Paulo",
    "sao paulo": "Sao Paulo",
    "sao paulo / sao paulo": "Sao Paulo",
    "sao paulop" : "Sao Paulo",
    "sao paulo - sp" : "Sao Paulo",
    "sp" : "Sao Paulo",
    "sao  paulo" : "Sao Paulo",
    "são paulo" : "Sao Paulo",
    "sao paluo" : "Sao Paulo",
    "rio de janeiro / rio de janeiro" : "Rio de Janeiro",
    "rio de janeiro" : "Rio de Janeiro",
    "rio de janeiro - rj" : "Rio de Janeiro",
    "rj" : "Rio de Janeiro",
    "rio de janeiro /rio de janeiro" :"Rio de Janeiro",
    "sao jose dos pinhais":"Sao Jose dos Pinhais",
    "sao jose dos pinhas":"Sao Jose dos Pinhais",
    "sao  jose dos pinhais" : "Sao Jose dos Pinhais",
    "balenario camboriu": "Balneario Camboriu",
    "balneario camboriu" : "Balneario Camboriu"
}

seller_temp = seller_temp.replace(city_dict_2, subset = ["seller_city"])
seller_temp.display()

In [0]:
seller_temp = seller_temp.withColumn("seller_city", F.initcap(F.col("seller_city")))
seller_temp.display()

In [0]:
seller3 = seller_temp.select(F.col("seller_city")).distinct()
seller3.display()

In [0]:
seller_temp = seller_temp.withColumn(
    "seller_city",
    F.when(
        F.col("seller_city").rlike(r"[/\\]"),
        "Multiple Cities"
    ).otherwise(F.col("seller_city"))
)
seller_temp.display()

In [0]:
seller_temp.write.mode("overwrite").saveAsTable("workspace.silver.silver_sellers")


Gold Layer

In [0]:
s_orders = spark.read.table("workspace.silver.silver_orders")
s_orders.display()
s_product = spark.read.table("workspace.silver.silver_product")
s_product.display()
s_seller = spark.read.table("workspace.silver.silver_sellers")
s_seller.display()
s_oi = spark.read.table("workspace.silver.silver_order_items")
s_oi.display()



In [0]:
gold_sales1 = s_oi\
    .join(
        s_orders, on = "order_id", how = "left")\
    .join(
        s_product, on = "product_id", how = "left")\
    .join(
        s_seller, on = "seller_id", how = "left")\
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_cancellation_stage",
        "product_category_name",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "seller_city",
        "seller_state"
    
        )

gold_sales1.display()

In [0]:
from pyspark.sql import functions as F
gold_sales1 = gold_sales1.withColumn("item_total_value", F.round( F.col("price") + F.col("freight_value"),2))


In [0]:
gold_sales1.show()

In [0]:
gold_sales1 = gold_sales1.withColumn("order_purchase_date",F.to_date(F.col("order_purchase_timestamp")))

gold_sales1 = gold_sales1.withColumn("order_purchase_month",F.month(F.col("order_purchase_date")))

gold_sales1.display()

In [0]:
gold_sales1.write.mode("overwrite").saveAsTable("workspace.gold.gold_sales")

In [0]:
cust_temp = spark.read.table("workspace.silver.silver_customer")
cust_temp.display()


In [0]:
gold_sales = spark.read.table("workspace.gold.gold_sales")
gold_sales_temp = gold_sales\
        .join(
            cust_temp.select(
            "customer_id",
            "customer_unique_id"
        ),on = "customer_id", how = "left")
gold_sales_temp.display()

In [0]:
gold_sales_temp.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.gold_sales")

In [0]:
customer_metrics = (
    gold_sales_temp
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("order_item_id").alias("total_items"),
        F.round(F.sum("price"),2).alias("total_spend"),
        F.round(F.sum("freight_value"),2).alias("total_freight"),
        F.round(F.sum("item_total_value"),2).alias("total_order_value"),
        F.countDistinct(
            F.when(
                F.col("order_status") == "canceled",
                F.col("order_id")
            )
        ).alias("canceled_orders"),
        F.min("order_purchase_date").alias("first_order_date"),
        F.max("order_purchase_date").alias("last_order_date")
    )
)

customer_metrics.display()

In [0]:
customer_metrics = customer_metrics.withColumn("Avg_order_value",F.round(F.col("total_order_value")/F.col("total_orders"),2))
customer_metrics.display()


In [0]:
customer_info = (
    cust_temp
    .groupBy("customer_unique_id")
    .agg(
        F.first("customer_city", ignorenulls=True).alias("customer_city"),
        F.first("customer_state", ignorenulls=True).alias("customer_state"),
        F.first("customer_zip_code_prefix", ignorenulls=True).alias("customer_zip_code_prefix")
    )
)
customer_info.display()

In [0]:
gold_customers = (
    customer_info
    .join(
        customer_metrics,
        on="customer_unique_id",
        how="left"
    )
)

gold_customers.display()

In [0]:
gold_customers.write.mode("overwrite").saveAsTable("workspace.gold.gold_customers")

In [0]:
silver_o_items = spark.read.table("workspace.silver.silver_order_items")
silver_o_items.display()

In [0]:
silver_products = spark.read.table("workspace.silver.silver_product")
silver_products.display()

In [0]:
silver_orders_review = spark.read.table("workspace.silver.silver_order_reviews")
silver_orders_review.display()

In [0]:
from pyspark.sql import functions as F
silver_reviews_temp = silver_orders_review.groupBy("order_id").agg(F.avg("review_score").alias("avg_review_score"))
silver_reviews_temp.display()


In [0]:
order_items_review = silver_o_items.join(silver_reviews_temp, "order_id")
order_items_review.display()

In [0]:
product_orderitem_review_temp = silver_products.join(order_items_review, "product_id")
product_orderitem_review_temp.display()

In [0]:
product_matrics = product_orderitem_review_temp.groupBy("product_id", "product_category_name").agg(
    F.count("order_item_id").alias("total_units_sold"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("price"), 2).alias("avg_product_price"),
        F.round(F.avg("freight_value"), 2).alias("avg_freight_value"),
        F.round(F.avg("avg_review_score"), 2).alias("avg_review_score")
)

product_matrics.display()

In [0]:
product_matrics.write.mode("overwrite").saveAsTable("workspace.gold.gold_product")

In [0]:
silver_order_temp = spark.read.table("workspace.silver.silver_orders")
silver_order_temp.display()

In [0]:
gold_dil_temp = silver_order_temp\
                .withColumn(
                    "delivery_days",
                    F.when(
                        F.col("order_delivered_customer_date").isNotNull(),
                        F.datediff(
                            F.to_date("order_delivered_customer_date"),
                            F.to_date("order_purchase_timestamp")
                            )
                        )
                    )\
                .withColumn(
                    "estimated_delivery_days",
                    F.datediff(
                        F.to_date("order_estimated_delivery_date"),
                        F.to_date("order_purchase_timestamp")
                        )
                    )\
                .withColumn(
                "delivery_delay_days",
                F.when(
                    F.col("order_delivered_customer_date").isNotNull()
                    & F.col("order_estimated_delivery_date").isNotNull(),
                    F.datediff(
                        F.to_date("order_delivered_customer_date"),
                        F.to_date("order_estimated_delivery_date")
                        )
                    )
                )
gold_dil_temp.display()
        

In [0]:
gold_dil_temp.write.mode("overwrite").saveAsTable("workspace.gold.gold_delivery")


In [0]:
s_o_pay = spark.read.table("workspace.silver.silver_order_payments")
s_o_pay.display()


In [0]:
Df_p1 = s_o_pay.select(
    "order_id",
    "payment_type",
    "payment_value",
    "payment_installments"
)

In [0]:
gold_payment_methods = (
    Df_p1
    .groupBy("order_id", "payment_type")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.count("*").alias("payment_records"),
        F.max("payment_installments").alias("max_installments")
    )
)

In [0]:
gold_payment_methods.display()

In [0]:
gold_payment_methods.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_payment_methods")

In [0]:
from pyspark.sql import functions as F
s_o_pay_temp = s_o_pay\
    .groupBy("order_id")\
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.count("*").alias("total_payment_records"),
        F.max("payment_installments").alias("max_installments"),
        F.countDistinct("payment_type").alias("payment_method_count")
        )
s_o_pay_temp.display()

In [0]:
s_o_pay_temp.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.gold_order_payments")

Gold Validation Checks


Gold_Sales_Test

In [0]:
from pyspark.sql import functions as F

sales = spark.read.table("workspace.gold.gold_sales")

# Row count
print("Sales rows:", sales.count())

# Duplicate order-item keys
sales.groupBy("order_id", "order_item_id") \    .count().filter(F.col("count") > 1).display()

# Nulls in important columns
sales.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in ["order_id", "order_item_id", "product_id",
              "customer_id", "price", "item_total_value"]
]).display()

# Check item_total_value = price + freight_value
sales.filter(
    F.abs(
        F.col("item_total_value") -
        (F.col("price") + F.col("freight_value"))
    ) > 0.01
).display()

Gold_customers_test



In [0]:
customers = spark.read.table("workspace.gold.gold_customers")

print("Customer rows:", customers.count())

# customer_unique_id must be unique
customers.groupBy("customer_unique_id") \
    .count().filter(F.col("count") > 1).display()

# Check null customer IDs
customers.filter(F.col("customer_unique_id").isNull()).display()

# Check invalid metric values
customers.filter(
    (F.col("total_orders") < 0) |
    (F.col("total_items") < 0) |
    (F.col("total_spend") < 0)
).display()

Gold_delivery_test


In [0]:
delivery = spark.read.table("workspace.gold.gold_delivery")

print("Delivery rows:", delivery.count())

# order_id should be unique
delivery.groupBy("order_id") \
    .count().filter(F.col("count") > 1).display()

# Check negative durations
delivery.filter(
    (F.col("delivery_days") < 0) |
    (F.col("estimated_delivery_days") < 0)
).display()

# Check delay calculation against the timestamps
delivery.filter(
    F.col("order_delivered_customer_date").isNotNull() &
    F.col("order_estimated_delivery_date").isNotNull() &
    (
        F.col("delivery_delay_days") != F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_estimated_delivery_date")
        )
    )
).display()

Gold_product_test

In [0]:
products = spark.read.table("workspace.gold.gold_product")

print("Product rows:", products.count())

# product_id must be unique
products.groupBy("product_id") \
    .count().filter(F.col("count") > 1).display()

# Check null IDs and negative metrics
products.filter(
    F.col("product_id").isNull() |
    (F.col("total_units_sold") < 0) |
    (F.col("total_revenue") < 0) |
    (F.col("avg_product_price") < 0)
).display()

# Review scores should normally be between 1 and 5
products.filter(
    F.col("avg_review_score").isNotNull() &
    ~F.col("avg_review_score").between(1, 5)
).display()

In [0]:
from pyspark.sql import functions as F

gold_delivery = spark.read.table("workspace.gold.gold_delivery")

customers = (
    spark.read.table("workspace.silver.silver_customer")
    .select("customer_id", "customer_state")
    .dropDuplicates(["customer_id"])
)

gold_delivery_updated = gold_delivery.join(
    customers,
    on="customer_id",
    how="left"
)

In [0]:
gold_delivery_updated.select(
    "order_id",
    "customer_id",
    "customer_state"
).show(10, truncate=False)

print("Original rows:", gold_delivery.count())
print("Updated rows:", gold_delivery_updated.count())

In [0]:
(
    gold_delivery_updated.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_delivery")
)

In [0]:
gold_tables = [
    "gold_sales",
    "gold_customers",
    "gold_product",
    "gold_delivery",
    "gold_order_payments",
    "gold_payment_methods"
]

In [0]:
for table in gold_tables:
    df = spark.read.table(f"workspace.gold.{table}")
    print(table, df.count())

In [0]:
import json
from google.cloud import bigquery
from google.oauth2 import service_account

# Retrieve credentials securely from Databricks Secrets
service_account_info = json.loads(
    dbutils.secrets.get(
        scope="gcp-credentials",
        key="gcp-service-account-json"
    )
)

# Authenticate with Google Cloud
credentials = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

# Create BigQuery client
client = bigquery.Client(
    project="databricksproject-510805",
    credentials=credentials
)

# Test access to destination dataset
dataset = client.get_dataset(
    "databricksproject-510805.db_olist_analytics"
)

print("BigQuery connection successful!")
print("Dataset:", dataset.dataset_id)

Gold transfer to Big Query From Databricks


In [0]:
%pip install google-cloud-bigquery


In [0]:
import json
from google.cloud import bigquery
from google.oauth2 import service_account

# Retrieve credentials securely from Databricks Secrets
service_account_info = json.loads(
    dbutils.secrets.get(
        scope="gcp-credentials",
        key="gcp-service-account-json"
    )
)

# Authenticate with Google Cloud
credentials = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

# Create BigQuery client
client = bigquery.Client(
    project="databricksproject-510805",
    credentials=credentials
)

# Test access to destination dataset
dataset = client.get_dataset(
    "databricksproject-510805.db_olist_analytics"
)

print("BigQuery connection successful!")
print("Dataset:", dataset.dataset_id)

In [0]:
df = spark.table("workspace.gold.gold_customers")

print("Databricks row count:", df.count())
display(df.limit(5))


In [0]:
from google.cloud import bigquery

# Convert Spark DataFrame to Pandas
pdf = df.toPandas()

# BigQuery destination
table_id = "databricksproject-510805.db_olist_analytics.gold_customers"

# Upload configuration
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Upload
job = client.load_table_from_dataframe(
    pdf,
    table_id,
    job_config=job_config
)

job.result()

print("Upload successful!")
print("Rows uploaded:", len(pdf))

In [0]:
table = client.get_table(
    "databricksproject-510805.db_olist_analytics.gold_customers"
)

print("BigQuery row count:", table.num_rows)
print("BigQuery columns:", len(table.schema))

In [0]:
from google.cloud import bigquery

def upload_to_bigquery(source_table, destination_table):

    # Read from Databricks Gold
    df = spark.table(f"workspace.gold.{source_table}")

    # Convert to Pandas
    pdf = df.toPandas()

    # BigQuery destination
    table_id = (
        f"databricksproject-510805.db_olist_analytics.{destination_table}"
    )

    # Upload configuration
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    )

    # Upload to BigQuery
    job = client.load_table_from_dataframe(
        pdf,
        table_id,
        job_config=job_config
    )

    job.result()

    print(f"Uploaded: {destination_table} | Rows: {len(pdf)}")


# Upload remaining Gold tables
tables = [
    ("gold_sales", "gold_sales"),
    ("gold_product", "gold_product"),
    ("gold_delivery", "gold_delivery"),
    ("gold_order_payments", "gold_order_payments"),
    ("gold_payment_methods", "gold_payment_methods")
]

for source, destination in tables:
    upload_to_bigquery(source, destination)

DATA VALIDATION


In [0]:
tables = {
    "gold_sales": 112650,
    "gold_customers": 96096,
    "gold_product": 32789,
    "gold_delivery": 99441,
    "gold_order_payments": 99440,
    "gold_payment_methods": 101686
}

for name, expected in tables.items():

    table_id = f"databricksproject-510805.db_olist_analytics.{name}"

    actual = client.get_table(table_id).num_rows

    status = "PASS" if actual == expected else "FAIL"

    print(f"{name}: {actual} | Expected: {expected} | {status}")

In [0]:
import json
from google.cloud import bigquery
from google.oauth2 import service_account

service_account_info = json.loads(
    dbutils.secrets.get(
        scope="gcp-credentials",
        key="gcp-service-account-json"
    )
)

credentials = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

client = bigquery.Client(
    project="databricksproject-510805",
    credentials=credentials
)

print("BigQuery client initialized successfully!")

In [0]:
tables = {
    "gold_sales": 112650,
    "gold_customers": 96096,
    "gold_product": 32789,
    "gold_delivery": 99441,
    "gold_order_payments": 99440,
    "gold_payment_methods": 101686
}

for name, expected in tables.items():

    table_id = f"databricksproject-510805.db_olist_analytics.{name}"

    actual = client.get_table(table_id).num_rows

    status = "PASS" if actual == expected else "FAIL"

    print(f"{name}: {actual} | Expected: {expected} | {status}")

In [0]:
checks = {
    "gold_sales": ["order_id", "order_item_id"],
    "gold_customers": ["customer_unique_id"],
    "gold_product": ["product_id"],
    "gold_delivery": ["order_id"],
    "gold_order_payments": ["order_id"],
    "gold_payment_methods": ["order_id", "payment_type"]
}

for table, keys in checks.items():
    df = spark.table(f"workspace.gold.{table}")

    total = df.count()
    distinct = df.select(*keys).distinct().count()

    print(
        f"{table}:",
        "PASS" if total == distinct else "FAIL",
        f"| Rows: {total} | Unique keys: {distinct}"
    )

In [0]:
# Databricks Gold
spark.table("workspace.gold.gold_sales").selectExpr(
    "ROUND(SUM(price), 2) AS total_revenue"
).show()